# Capítulo 2 - Exercício 2: Substituindo `GridSearchCV` por `RandomizedSearchCV`

O objetivo deste notebook é continuar o teste com `SVR`, mas trocar a busca em grade por uma busca aleatória. Em vez de testar uma grade fixa de valores, vamos sortear combinações de hiperparâmetros a partir de distribuições.

## Plano do exercício

1. Carregar os dados brutos.
2. Recriar a divisão estratificada treino/teste usada no capítulo.
3. Separar atributos e alvo.
4. Montar o mesmo pré-processamento usado no projeto principal.
5. Criar um pipeline com `SVR`.
6. Usar `RandomizedSearchCV` com distribuições para `C` e `gamma`.
7. Avaliar o melhor modelo no conjunto de teste.

## Configuração

Importamos as bibliotecas usadas no exercício e verificamos versões mínimas. Mantemos uma semente fixa para tornar os resultados mais reprodutíveis.

In [25]:
import sys
from pathlib import Path
import tarfile
import urllib.request

import numpy as np
import pandas as pd

from packaging import version
import sklearn

assert sys.version_info >= (3, 7)
assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

np.random.seed(42)

## Carregando os dados

Esta função é a mesma ideia usada no notebook principal: se o arquivo `housing.tgz` não existir, ele é baixado; em seguida o CSV é extraído e carregado em um `DataFrame`.

In [26]:
def load_housing_data():
    tarball_path = Path("datasets/housing.tgz")
    if not tarball_path.is_file():
        Path("datasets").mkdir(parents=True, exist_ok=True)
        url = "https://github.com/ageron/data/raw/main/housing.tgz"
        urllib.request.urlretrieve(url, tarball_path)
    with tarfile.open(tarball_path) as housing_tarball:
        housing_tarball.extractall(path="datasets")
    return pd.read_csv(Path("datasets/housing/housing.csv"))

housing = load_housing_data()
housing.head()

/tmp/ipykernel_221944/2931591054.py:8: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  housing_tarball.extractall(path="datasets")


,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## Divisão treino/teste estratificada

Como no capítulo, criamos categorias de renda para garantir que treino e teste mantenham proporções parecidas de `median_income`. Depois removemos a coluna auxiliar `income_cat`, pois ela só serve para a amostragem.

In [27]:
from sklearn.model_selection import train_test_split

housing["income_cat"] = pd.cut(
    housing["median_income"],
    bins=[0., 1.5, 3.0, 4.5, 6., np.inf],
    labels=[1, 2, 3, 4, 5]
)

strat_train_set, strat_test_set = train_test_split(
    housing,
    test_size=0.2,
    stratify=housing["income_cat"],
    random_state=42
)

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

housing.shape, strat_test_set.shape

((16512, 9), (4128, 10))

## Pré-processamento

Abaixo recriamos o pipeline completo do notebook principal: razões entre atributos, transformação logarítmica, similaridade geográfica via clusters, tratamento de categorias e escalonamento. Assim o `SVR` recebe os mesmos atributos preparados que os modelos anteriores.

In [28]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler


def column_ratio(X):
    return X[:, [0]] / X[:, [1]]


def ratio_name(function_transformer, feature_names_in):
    return ["ratio"]


def ratio_pipeline():
    return make_pipeline(
        SimpleImputer(strategy="median"),
        FunctionTransformer(column_ratio, feature_names_out=ratio_name),
        StandardScaler()
    )


class ClusterSimilarity(BaseEstimator, TransformerMixin):
    def __init__(self, n_clusters=10, gamma=1.0, random_state=None):
        self.n_clusters = n_clusters
        self.gamma = gamma
        self.random_state = random_state

    def fit(self, X, y=None, sample_weight=None):
        self.kmeans_ = KMeans(self.n_clusters, n_init=10,
                              random_state=self.random_state)
        self.kmeans_.fit(X, sample_weight=sample_weight)
        return self

    def transform(self, X):
        return rbf_kernel(X, self.kmeans_.cluster_centers_, gamma=self.gamma)

    def get_feature_names_out(self, names=None):
        return [f"Similaridade com o cluster {i}" for i in range(self.n_clusters)]


log_pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    FunctionTransformer(np.log, feature_names_out="one-to-one"),
    StandardScaler()
)

cat_pipeline = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

cluster_simil = ClusterSimilarity(n_clusters=10, gamma=1., random_state=42)
default_num_pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler())

preprocessing = ColumnTransformer([
    ("bedrooms", ratio_pipeline(), ["total_bedrooms", "total_rooms"]),
    ("rooms_per_house", ratio_pipeline(), ["total_rooms", "households"]),
    ("people_per_house", ratio_pipeline(), ["population", "households"]),
    ("log", log_pipeline, ["total_bedrooms", "total_rooms", "population",
                           "households", "median_income"]),
    ("geo", cluster_simil, ["latitude", "longitude"]),
    ("cat", cat_pipeline, make_column_selector(dtype_include=object)),
], remainder=default_num_pipeline)

## Métrica de avaliação

Usaremos RMSE, a mesma métrica do projeto principal. O bloco abaixo funciona tanto em versões novas quanto antigas do Scikit-Learn.

In [29]:
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error

    def root_mean_squared_error(labels, predictions):
        return mean_squared_error(labels, predictions, squared=False)

## Treinando um `SVR` com busca aleatória

No exercício anterior, a busca em grade testava valores fixos. Aqui usamos `RandomizedSearchCV`: `C` será sorteado em escala logarítmica com `loguniform`, e `gamma` será sorteado com `expon`. Isso permite explorar um espaço maior de valores sem testar todas as combinações possíveis.

Como `SVR` pode ser lento neste conjunto de dados, seguimos a ideia do livro e ajustamos a busca usando apenas as primeiras 5.000 instâncias do treino.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import expon, loguniform
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR

svr_pipeline = Pipeline([
    ("preprocessing", preprocessing),
    ("svr", SVR()),
])

param_distribs = {
    "svr__kernel": ["linear", "rbf"],
    "svr__C": loguniform(20, 200_000),
    "svr__gamma": expon(scale=1.0),
}

svr_rnd_search = RandomizedSearchCV(
    svr_pipeline,
    param_distributions=param_distribs,
    n_iter=50,
    cv=3,
    scoring="neg_root_mean_squared_error",
    random_state=42,
    n_jobs=-1,
    verbose=2,
)

svr_rnd_search.fit(housing.iloc[:5000], housing_labels.iloc[:5000])

Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END svr__C=432.37884813148844, svr__gamma=0.15416196746656105, svr__kernel=linear; total time=   0.5s
[CV] END svr__C=629.7823295913721, svr__gamma=3.010121430917521, svr__kernel=linear; total time=   0.5s
[CV] END svr__C=629.7823295913721, svr__gamma=3.010121430917521, svr__kernel=linear; total time=   0.5s
[CV] END svr__C=432.37884813148844, svr__gamma=0.15416196746656105, svr__kernel=linear; total time=   0.5s
[CV] END svr__C=432.37884813148844, svr__gamma=0.15416196746656105, svr__kernel=linear; total time=   0.5s
[CV] END svr__C=629.7823295913721, svr__gamma=3.010121430917521, svr__kernel=linear; total time=   0.6s
[CV] END svr__C=84.14107900575871, svr__gamma=0.059838768608680676, svr__kernel=rbf; total time=   0.8s
[CV] END svr__C=84.14107900575871, svr__gamma=0.059838768608680676, svr__kernel=rbf; total time=   0.8s
[CV] END svr__C=26290.20646430022, svr__gamma=0.9084469696321253, svr__kernel=rbf; total time=   

RandomizedSearchCV(cv=3,
                   estimator=Pipeline(steps=[('preprocessing',
                                              ColumnTransformer(remainder=Pipeline(steps=[('simpleimputer',
                                                                                           SimpleImputer(strategy='median')),
                                                                                          ('standardscaler',
                                                                                           StandardScaler())]),
                                                                transformers=[('bedrooms',
                                                                               Pipeline(steps=[('simpleimputer',
                                                                                                SimpleImputer(strategy='median')),
                                                                                               ('functiontransformer',
                                                                                                FunctionTransformer(feature_names_...
                                                                               <sklearn.compose._column_transformer.make_column_selector object at 0x7f24ec22c0b0>)])),
                                             ('svr', SVR())]),
                   n_iter=50, n_jobs=-1,
                   param_distributions={'svr__C': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7f24ec090260>,
                                        'svr__gamma': <scipy.stats._distn_infrastructure.rv_continuous_frozen object at 0x7f24ec24f0b0>,
                                        'svr__kernel': ['linear', 'rbf']},
                   random_state=42, scoring='neg_root_mean_squared_error',
                   verbose=2)

## Resultados da busca

Aqui organizamos os resultados da validação cruzada. Como o Scikit-Learn usa pontuação negativa para RMSE, multiplicamos por `-1` para facilitar a leitura.

In [31]:
svr_cv_results = pd.DataFrame(svr_rnd_search.cv_results_)
svr_cv_results["mean_test_rmse"] = -svr_cv_results["mean_test_score"]

cols = ["mean_test_rmse", "param_svr__kernel", "param_svr__C", "param_svr__gamma"]
svr_cv_results[cols].sort_values("mean_test_rmse").head(10)

,mean_test_rmse,param_svr__kernel,param_svr__C,param_svr__gamma
9,55805.373324,rbf,157055.109894,0.264970
28,58276.929873,rbf,24547.601976,0.221539
37,61533.897450,rbf,15415.161545,0.269168
8,65848.355468,rbf,5603.270317,0.150235
31,68534.014480,rbf,101445.668813,1.052904
30,69755.267014,linear,16483.850530,1.475215
10,70030.182322,linear,27652.464359,0.222736
45,70195.592858,linear,6287.039489,0.350457
14,70217.132961,linear,34246.751946,0.363288
48,70221.886236,linear,33946.157065,2.264243


In [32]:
svr_rnd_search.best_params_, -svr_rnd_search.best_score_

({'svr__C': np.float64(157055.10989448498),
  'svr__gamma': np.float64(0.26497040005002437),
  'svr__kernel': 'rbf'},
 np.float64(55805.37332417717))

## Avaliação no conjunto de teste

Depois de escolher o melhor `SVR` pela validação cruzada, avaliamos uma única vez no conjunto de teste. Como a busca foi feita em uma amostra de 5.000 instâncias, esse resultado pode variar em relação a uma busca mais longa com todo o treino.

In [33]:
X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

best_svr_model = svr_rnd_search.best_estimator_
svr_test_predictions = best_svr_model.predict(X_test)
svr_test_rmse = root_mean_squared_error(y_test, svr_test_predictions)

svr_test_rmse

np.float64(57198.310118484704)